# Regresión Lineal Múltiple Aplicada: Variables Categóricas, Ordinales y Continuas

Este notebook muestra cómo tratar variables independientes de diferentes tipos (categóricas, ordinales y continuas) para predecir una variable dependiente continua usando un dataset real de negocio.

## 1. Importar librerías necesarias

Importamos las librerías para manipulación de datos, visualización y modelado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# Configuración de gráficos
sns.set_theme(style="whitegrid")

## 2. Cargar y explorar un dataset real de negocio

Usaremos el dataset **Hotel Booking Demand** que contiene reservas de hoteles con información de pricing (ADR: Average Daily Rate), tipos de cliente, canales de distribución y características de la reserva.

In [ ]:
# Cargar el dataset California Housing desde sklearn
from sklearn.datasets import fetch_california_housing

california = fetch_california_housing(as_frame=True)
df = california.frame

# Creamos variables categóricas y ordinales para ilustrar el tratamiento
# Categórica nominal: Región basada en latitud
df["Region"] = np.where(df["Latitude"] > 36, "Norte", "Sur")

# Ordinal: Nivel de antigüedad de la vivienda
df["Antiguedad"] = pd.cut(
    df["HouseAge"],
    bins=[0, 15, 35, 100],
    labels=["Nueva", "Media", "Antigua"],
    ordered=True,
)

# Mostramos las primeras filas
print(f"Dataset: {df.shape[0]} viviendas, {df.shape[1]} variables")
df.head()

## 3. Preprocesamiento y análisis de variables

Identificamos y tratamos las variables categóricas, ordinales y continuas. También revisamos valores nulos y tipos de datos.

In [ ]:
# Revisar tipos de variables y valores nulos
print(df.info())
print("\nValores nulos por columna:\n", df.isnull().sum())

# Identificar variables por tipo
cat_vars = ["Region"]  # Categórica nominal

# Variable ordinal: Antigüedad con orden natural
antiguedad_order = {"Nueva": 0, "Media": 1, "Antigua": 2}
df["Antiguedad_ord"] = df["Antiguedad"].map(antiguedad_order)

# Variables ordinales/continuas discretas
ord_vars = ["Antiguedad_ord"]

# Variables continuas
cont_vars = [
    "MedInc",
    "AveRooms",
    "AveBedrms",
    "Population",
    "AveOccup",
    "Latitude",
    "Longitude",
]

# Variable objetivo
target = "MedHouseVal"

print(f"\nCategóricas: {cat_vars}")
print(f"Ordinales: {ord_vars}")
print(f"Continuas: {cont_vars}")
print(f"Objetivo: {target}")

df[cat_vars + ["Antiguedad", "Antiguedad_ord", target]].head()

## 4. Análisis exploratorio de datos (EDA)

Visualizamos la distribución de variables y relaciones entre ellas.

In [ ]:
# Histogramas de variables numéricas
num_for_plot = cont_vars + ord_vars + [target]
_ = df[num_for_plot].hist(bins=20, figsize=(12, 8))
plt.suptitle("Distribución de variables continuas")
plt.tight_layout()
plt.show()

# Boxplot de la variable dependiente por Región
plt.figure(figsize=(6, 4))
sns.boxplot(x="Region", y=target, data=df)
plt.title("Valor mediano de vivienda por Región")
plt.ylabel("Precio (x$100k)")
plt.show()

# Boxplot de la variable dependiente por Antigüedad
plt.figure(figsize=(6, 4))
sns.boxplot(x="Antiguedad", y=target, data=df, order=["Nueva", "Media", "Antigua"])
plt.title("Valor mediano de vivienda por Antigüedad")
plt.ylabel("Precio (x$100k)")
plt.show()

# Mapa de calor de correlación
plt.figure(figsize=(10, 8))
sns.heatmap(df[num_for_plot].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlación entre variables numéricas")
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot de variables clave (submuestra para rendimiento)
cols_pp = [
    target,
    "MedInc",
    "AveRooms",
    "AveBedrms",
    "Population",
    "AveOccup",
    "Latitude",
]

# Submuestreamos para que el gráfico sea manejable en clase
sample_df = df.sample(n=3000, random_state=42) if len(df) > 3000 else df.copy()

pp = sns.pairplot(
    data=sample_df,
    vars=cols_pp,
    hue="Region",
    diag_kind="kde",
    corner=True,
    plot_kws={"alpha": 0.4, "s": 12},
)
pp.figure.suptitle("Pairplot de variables clave (hue=Region, submuestreo)", y=1.02)
plt.show()

## 5. Codificación de variables categóricas y ordinales

Aplicamos One-Hot Encoding para la variable categórica y Ordinal Encoding para la variable ordinal.

In [ ]:
# One-Hot Encoding para Región (categórica nominal)
region_encoded = pd.get_dummies(
    df["Region"], prefix="Region", drop_first=True, dtype=int
)

# Variable ordinal ya codificada (Antiguedad_ord)
df["Antiguedad_ord"] = df["Antiguedad_ord"].fillna(0).astype(int)

# Unimos todo en un nuevo DataFrame para modelar
X = pd.concat([df[cont_vars + ord_vars], region_encoded], axis=1)

# Aseguramos que todas las columnas sean numéricas
X = X.apply(pd.to_numeric, errors="coerce").astype(float)

y = df[target]

print(f"Variables predictoras: {X.shape[1]}")
print(f"Muestras: {X.shape[0]}")
X.head()

## 6. Modelo de regresión lineal simple (baseline)

Creamos un modelo simple usando solo una variable predictora para comparar con el modelo múltiple.

In [ ]:
# Usamos solo 'MedInc' (ingreso mediano) como predictor baseline
X_simple = df[["MedInc"]]
y_simple = df[target]

model_simple = LinearRegression()
model_simple.fit(X_simple, y_simple)

# Predicción y métricas
y_pred_s = model_simple.predict(X_simple)
print("Modelo Simple (solo MedInc):")
print("  R² =", round(r2_score(y_simple, y_pred_s), 4))
print("  RMSE =", round(np.sqrt(mean_squared_error(y_simple, y_pred_s)), 2), "x$100k")

## 7. Modelo de regresión lineal múltiple

Incluimos variables continuas, categóricas y ordinales. Separamos en train y test.

In [ ]:
# Modelo múltiple con todos los datos
model_multi = LinearRegression()
model_multi.fit(X, y)

y_pred = model_multi.predict(X)

print("Modelo Múltiple (todas las variables):")
print("  R² =", round(r2_score(y, y_pred), 4))
print("  RMSE =", round(np.sqrt(mean_squared_error(y, y_pred)), 2), "x$100k")

## 8. Diagnóstico del modelo y supuestos

Comprobamos los supuestos de la regresión: linealidad, normalidad de residuos, homocedasticidad e independencia.

In [ ]:
# Residuos
residuals = y - y_pred

# Linealidad y homocedasticidad
plt.figure(figsize=(6, 4))
plt.scatter(y_pred, residuals, alpha=0.3)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicción")
plt.ylabel("Residuo")
plt.title("Residuos vs Predicción")
plt.show()

# Normalidad de residuos
sm.qqplot(residuals, line="s")
plt.title("Q-Q plot de residuos")
plt.show()

# Histograma de residuos
plt.hist(residuals, bins=30)
plt.title("Histograma de residuos")
plt.xlabel("Residuo")
plt.show()

## 9. Comprobación de multicolinealidad

Calculamos el VIF para cada predictor y revisamos la matriz de correlación.

In [ ]:
# VIF para cada predictor (asegurando tipos numéricos y datos válidos)
X_vif = sm.add_constant(X)

# Convertir todo a float y limpiar datos problemáticos
X_vif = X_vif.apply(pd.to_numeric, errors="coerce").astype(float)
# Eliminar columnas constantes que rompen el cálculo de VIF
constant_cols = [col for col in X_vif.columns if X_vif[col].nunique() <= 1]
if constant_cols:
    X_vif = X_vif.drop(columns=constant_cols)

# Eliminar filas con NaN o infinitos
X_vif = X_vif.replace([np.inf, -np.inf], np.nan).dropna(axis=0)

features = list(X_vif.columns)
# Calcular VIF excluyendo la constante si existe
vif_list = []
feat_list = []
for i, col in enumerate(features):
    if col == "const":
        continue
    vif_list.append(variance_inflation_factor(X_vif.values, i))
    feat_list.append(col)

vif_data = pd.DataFrame({"feature": feat_list, "VIF": vif_list}).sort_values(
    "VIF", ascending=False
)
vif_data

## 10. Interpretación de coeficientes

Analizamos los coeficientes, errores estándar y significación estadística del modelo múltiple.

In [ ]:
# Usamos statsmodels para obtener resumen completo
# Aseguramos que todas las columnas sean numéricas
X_clean = X.apply(pd.to_numeric, errors="coerce")
X_sm = sm.add_constant(X_clean)
model_sm = sm.OLS(y, X_sm).fit()
model_sm.summary()

## 11. Comparación y selección de modelos

Comparamos el modelo simple y el múltiple usando R², R² ajustado y métricas de test.

In [ ]:
# Comparación de modelos
print("Comparación de modelos:")
print("=" * 50)
print(f"R² simple:              {round(r2_score(y_simple, y_pred_s), 4)}")
print(f"R² múltiple:            {round(r2_score(y, y_pred), 4)}")
print(f"R² ajustado múltiple:   {round(model_sm.rsquared_adj, 4)}")
print(
    f"RMSE simple:            {round(np.sqrt(mean_squared_error(y_simple, y_pred_s)), 2)} x$100k"
)
print(
    f"RMSE múltiple:          {round(np.sqrt(mean_squared_error(y, y_pred)), 2)} x$100k"
)
print(f"MAE múltiple:           {round(mean_absolute_error(y, y_pred), 2)} x$100k")